# 🌿 مشروع AgriTech: مشغّل ومختبِر النموذج المباشر في Google Colab
### تشخيص أمراض أوراق البن + الرفض الذكي (Smart Rejection)
**إعداد الطالبة:** حواء  
**النموذج:** `final.h5` (MobileNetV2 - 5 فئات معتمدة)  

---
### 📌 كيفية الاستخدام في ثوانٍ:
1. شغّلي **الخلية الأولى أدناه**: ستقوم بتحميل النموذج تلقائياً (أو تطلب رفعه إذا لم يكن مرفوعاً)، ثم تفتح لكِ نافذة لاختيار صورة ورقة البن.
2. فور اختيار الصورة، سيظهر التشخيص والرسم البياني الملون وقرار الرفض الذكي مباشرة!
3. لتجربة أي صورة أخرى لاحقاً، شغّلي **الخلية الثانية** دون الحاجة لإعادة تحميل النموذج.

In [ ]:
# ==============================================================================
# الخلية 1: تشغيل محرك الذكاء الاصطناعي واختبار أي صورة مباشرة (All-In-One)
# ==============================================================================
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

# 1. حل مشكلة Keras 3 في Colab نهائياً عبر محرك tf_keras الرسمي
try:
    import tf_keras as keras
except ImportError:
    print('⏳ تثبيت حزمة التوافق tf_keras...')
    os.system('pip install -q tf_keras')
    import tf_keras as keras

import tensorflow as tf
print(f'✓ إصدار TensorFlow: {tf.__version__}')

# 2. التحقق من وجود ملف النموذج final.h5 أو طلبه فوراً
MODEL_NAME = 'final.h5'
model_path = None
for p in [MODEL_NAME, f'/content/{MODEL_NAME}', 'best_coffee_model.h5']:
    if os.path.exists(p):
        model_path = p
        break

if model_path is None:
    print(f'\n⚠️ لم يتم العثور على {MODEL_NAME} في مساحة العمل.')
    print('📤 يرجى الضغط على الزر أدناه واختيار ملف final.h5 من جهازك (~19MB):')
    uploaded_m = files.upload()
    for fname in uploaded_m.keys():
        if fname.endswith(('.h5', '.keras')):
            model_path = fname
            break

if model_path is None or not os.path.exists(model_path):
    raise FileNotFoundError('❌ لم يتم العثور على ملف النموذج! يرجى التأكد من اختياره عبر زر الرفع.')

print(f'\n⏳ جاري تحميل النموذج من: {model_path} ...')
try:
    model = keras.models.load_model(model_path, compile=False)
except Exception:
    from tensorflow.keras.layers import DepthwiseConv2D
    class PatchedDepthwiseConv2D(DepthwiseConv2D):
        def __init__(self, *args, **kwargs):
            kwargs.pop('groups', None)
            super().__init__(*args, **kwargs)
    model = tf.keras.models.load_model(model_path, compile=False, custom_objects={'DepthwiseConv2D': PatchedDepthwiseConv2D})

print(f'✅ تم تحميل النموذج بنجاح تام! (عدد المعاملات: {model.count_params():,})')

# 3. تعريف الفئات وقاعدة المعرفة الزراعية والتشخيصية
CLASS_NAMES = ['Cerscospora', 'Healthy', 'Leaf rust', 'Miner', 'Phoma']

DISEASE_INFO = {
    'Cerscospora': {'ar': 'التبقع البني (عين الطائر)', 'color': '#d9534f', 'advice': 'تنظيم فترات الري، تقليم الأغصان المتشابكة، وتطبيق مبيد فطري نحاسي وقائي.'},
    'Healthy': {'ar': 'ورقة سليمة وخالية من الإصابة', 'color': '#2e7d32', 'advice': 'حالة مثالية؛ يوصى بمواصلة التسميد المتوازن والري المنتظم.'},
    'Leaf rust': {'ar': 'صدأ أوراق البن (خطير)', 'color': '#f57c00', 'advice': 'عزل الأوراق المصابة فوراً والرش بمبيد فطري جهازي (مركبات التريازول).'},
    'Miner': {'ar': 'حافرة أنفاق أوراق البن', 'color': '#795548', 'advice': 'إزالة الأوراق ذات الأنفاق النشطة واستخدام مكافحة حشرية متخصصة.'},
    'Phoma': {'ar': 'اللفحة الفومية (التبقع الفومي)', 'color': '#5c6bc0', 'advice': 'تقليم الأغصان الميتة والتهوية، والرش الوقائي بعد موجات الرياح والبرد.'}
}

def diagnose_and_plot(img_path):
    with Image.open(img_path) as img:
        img_rgb = img.convert('RGB')
        w, h = img_rgb.size
        min_dim = min(w, h)
        left = (w - min_dim) // 2
        top = (h - min_dim) // 2
        img_cropped = img_rgb.crop((left, top, left + min_dim, top + min_dim))
        img_resized = img_cropped.resize((224, 224), Image.BILINEAR)
        img_arr = np.array(img_resized, dtype=np.float32)
        img_preprocessed = (img_arr / 127.5) - 1.0
        batch = np.expand_dims(img_preprocessed, axis=0)

    probs = model.predict(batch, verbose=0)[0]
    top_idx = int(np.argmax(probs))
    top_class = CLASS_NAMES[top_idx]
    confidence = float(probs[top_idx]) * 100.0
    info = DISEASE_INFO[top_class]

    print('=' * 68)
    print(f'🌿 نتيجة تشخيص ورقة البن: {os.path.basename(img_path)}')
    print('=' * 68)
    print(f'  • المرض المشخّص:  {info["ar"]}')
    print(f'  • نسبة الثقة:      {confidence:.2f}%')

    if confidence < 85.0:
        print('\n  ⚠️ تنبيه الرفض الذكي (Smart Rejection):')
        print('  الأعراض غير واضحة أو في مرحلة مبكرة جداً. يرجى الاقتراب أكثر من البقعة المصابة وإعادة التصوير.')
    else:
        print(f'  • التوصية الزراعية: {info["advice"]}')
    print('=' * 68 + '\n')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.5))
    ax1.imshow(img_cropped)
    status_text = '[مقبول - مؤكد]' if confidence >= 85.0 else '[مرفوض - أعراض غير واضحة]'
    status_col = info['color'] if confidence >= 85.0 else '#d9534f'
    ax1.set_title(f'{top_class} ({confidence:.1f}%)\n{status_text}', fontsize=12, fontweight='bold', color=status_col)
    ax1.axis('off')

    bar_colors = [info['color'] if i == top_idx else '#cfd8dc' for i in range(len(CLASS_NAMES))]
    bars = ax2.barh(CLASS_NAMES, probs * 100.0, color=bar_colors, height=0.55)
    ax2.set_xlim(0, 100)
    ax2.set_xlabel('احتمالية الفئة (%)', fontsize=10, fontweight='bold')
    ax2.set_title('توزيع الاحتمالات عبر فئات أمراض البن الخمس', fontsize=11, fontweight='bold')
    ax2.grid(axis='x', linestyle='--', alpha=0.5)

    for bar in bars:
        w = bar.get_width()
        ax2.text(w + 1.2, bar.get_y() + bar.get_height() / 2, f'{w:.1f}%', va='center', fontsize=9, fontweight='bold')

    plt.tight_layout()
    plt.show()

# 4. رفع واختبار صورة ورقة البن الآن
print('\n📸 يرجى الضغط على زر (Choose Files) أدناه واختيار صورة ورقة البن لاختبارها:')
uploaded_images = files.upload()
for fname in uploaded_images.keys():
    print(f'\n🔎 تحليل الصورة: {fname} ...')
    diagnose_and_plot(fname)


In [ ]:
# ==============================================================================
# الخلية 2: لفحص أي صورة إضافية بسرعة (بدون إعادة تحميل المودل)
# ==============================================================================
print('📸 اختاري صورة أخرى لورقة البن لفحصها:')
extra_images = files.upload()
for fname in extra_images.keys():
    diagnose_and_plot(fname)
